In [ ]:
# Ячейка 1: Импорты и настройки
%matplotlib inline
import re
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm # Автоматический выбор виджета для ноутбука
from clearml import Task, Logger
from IPython.display import clear_output

# Ячейка 2: Вспомогательные классы (Tokenizer, Dataset, Model)
class WikiTokenizer:
    def __init__(self, min_freq=5):
        self.itos = {0: "<unk>"}
        self.stoi = {"<unk>": 0}
        self.min_freq = min_freq

    def build_vocabulary(self, texts):
        freqs = Counter()
        for t in texts: 
            freqs.update(re.findall(r"[\w']+", t.lower()))
        idx = 1
        for word, freq in freqs.items():
            if freq >= self.min_freq:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1
    
    def encode(self, text):
        return [self.stoi.get(w, 0) for w in re.findall(r"[\w']+", text.lower())]
    
    def decode(self, tokens):
        return [self.itos.get(int(t), "<unk>") for t in tokens]

    def __len__(self):
        return len(self.itos)

class WikiDataset(Dataset):
    def __init__(self, texts, tokenizer, seq_len):
        self.chunks = []
        for t in texts:
            tokens = tokenizer.encode(t)
            for i in range(0, len(tokens) - seq_len, seq_len):
                chunk = torch.tensor(tokens[i:i+seq_len+1], dtype=torch.long)
                if len(chunk) == seq_len + 1:
                    self.chunks.append(chunk)
    def __len__(self): return len(self.chunks)
    def __getitem__(self, i): return self.chunks[i][:-1], self.chunks[i][1:]

class RNNModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, rnn_type, n_layers, drop):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        RNN = nn.LSTM if rnn_type == 'LSTM' else nn.GRU
        self.rnn = RNN(emb_dim, hid_dim, n_layers, batch_first=True, dropout=drop)
        self.fc = nn.Linear(hid_dim, vocab_size)
    
    def forward(self, x):
        out, _ = self.rnn(self.embedding(x))
        return self.fc(out)

# Ячейка 3: Основной класс Тренера
class RNNTrainer:
    def __init__(self, config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Загружаем данные один раз для всех экспериментов (можно вынести за класс для скорости)
        self._prepare_data()
        
        # Инициализация Task
        self.task = Task.init(
            project_name=config["project_name"],
            task_name=config["task_name"],
            reuse_last_task_id=False
        )
        self.task.connect(config)
        self.logger = Logger.current_logger()
        
        self.model = RNNModel(
            self.vocab_size, config["embedding_dim"], config["hidden_dim"],
            config["rnn_type"], config["num_layers"], config["dropout"]
        ).to(self.device)
        
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=config["lr"])
        self.criterion = nn.CrossEntropyLoss()

    def _prepare_data(self):
        dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
        # Простейшая очистка
        train_raw = [t for t in dataset['train']['text'] if len(t.split()) > self.config["seq_len"]]
        
        self.tokenizer = WikiTokenizer(min_freq=self.config["min_word_freq"])
        self.tokenizer.build_vocabulary(train_raw)
        self.vocab_size = len(self.tokenizer)
        
        train_ds = WikiDataset(train_raw, self.tokenizer, self.config["seq_len"])
        val_ds = WikiDataset(dataset['validation']['text'], self.tokenizer, self.config["seq_len"])
        
        self.train_loader = DataLoader(train_ds, batch_size=self.config["batch_size"], shuffle=True)
        self.val_loader = DataLoader(val_ds, batch_size=self.config["batch_size"])

    def train(self):
        best_loss = float('inf')
        patience_counter = 0
        
        for epoch in range(self.config["n_epochs"]):
            self.model.train()
            epoch_loss = 0
            
            pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}")
            for x, y in pbar:
                x, y = x.to(self.device), y.to(self.device)
                self.optimizer.zero_grad()
                logits = self.model(x)
                loss = self.criterion(logits.view(-1, self.vocab_size), y.view(-1))
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), self.config["max_norm"])
                self.optimizer.step()
                epoch_loss += loss.item()
                pbar.set_postfix(loss=loss.item())

            avg_train_loss = epoch_loss / len(self.train_loader)
            val_loss, val_ppl = self.evaluate()
            
            # ClearML Logs
            self.logger.report_scalar("Loss", "Train", avg_train_loss, epoch)
            self.logger.report_scalar("Loss", "Val", val_loss, epoch)
            self.logger.report_scalar("PPL", "Val", val_ppl, epoch)
            
            print(f"Summary Epoch {epoch}: Val Loss: {val_loss:.4f} | PPL: {val_ppl:.2f}")

            # Early Stopping
            if val_loss < best_loss:
                best_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= 3:
                    print("Early Stopping triggered!")
                    break
        
        self.task.close()

    def evaluate(self):
        self.model.eval()
        v_loss = 0
        with torch.no_grad():
            for x, y in self.val_loader:
                x, y = x.to(self.device), y.to(self.device)
                out = self.model(x)
                v_loss += self.criterion(out.view(-1, self.vocab_size), y.view(-1)).item()
        avg_v = v_loss / len(self.val_loader)
        return avg_v, np.exp(avg_v)

# Ячейка 4: Запуск цикла экспериментов
Task.set_credentials(
    api_host="https://api.clear.ml",
    web_host="https://app.clear.ml/",
    files_host="ttps://files.clear.ml",
    key="ARV0QJGZFMEMOF2GM6HM5A7SNKMWBJ",
    secret="ixgZR0oxLQqZUYR2TcHt_DFCnfU7M3RQMDb4KvL65eDOtPv9g9-pkvoHCkRwxOTyxP0"
)

import itertools
import gc
# Остальные импорты (torch, nn, Task и т.д.) остаются из предыдущего блока

# --- НАСТРОЙКА СЕТКИ ПАРАМЕТРОВ ---
search_space = {
    "lr": [0.001, 0.0005],
    "dropout": [0.1, 0.5],
    "min_word_freq": [2, 5],
    "max_norm": [1, 5],
}

# Базовый конфиг (неизменные параметры)
base_config = {
    "project_name": "WikiText_Global_Search",
    "rnn_type": ["GRU"],
    "n_epochs": 15,
    "weight_decay": 1e-5
    "hidden_dim": 1024,
    "embedding_dim": 512,
    "batch_size": 64,
    "seq_len": 32,
    "num_layers": 2
}

# Функция для генерации всех комбинаций
def get_experiment_list(space):
    keys, values = zip(*space.items())
    return [dict(zip(keys, v)) for v in itertools.product(*values)]

experiments = get_experiment_list(search_space)
print(f"Запланировано экспериментов: {len(experiments)}")

# --- ГЛОБАЛЬНЫЙ ЦИКЛ ЗАПУСКА ---

for i, params in enumerate(experiments):
    # Формируем итоговый конфиг для текущей итерации
    current_config = {**base_config, **params}
    
    # Красивое имя для ClearML, чтобы сразу видеть отличия в списке
    task_name = f"Exp_{i}_RNN_NEXT_WORD | {params['rnn_type']} | h{params['hidden_dim']} | lr{params['lr']}"
    current_config["task_name"] = task_name
    
    print(f"\n{'='*60}")
    print(f"ЗАПУСК {i+1}/{len(experiments)}: {task_name}")
    print(f"{'='*60}")
    
    try:
        # Инициализируем тренера (внутри создается Task)
        trainer = RNNTrainer(current_config)
        trainer.train()
        
        # После завершения обучения принудительно чистим память
        del trainer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        clear_output(wait=True) # Очищаем вывод в ноутбуке, чтобы не вис браузер
        
    except Exception as e:
        print(f"!!! Ошибка в эксперименте {task_name}: {e}")
        # Закрываем таск, если он успел инициализироваться
        current_task = Task.current_task()
        if current_task:
            current_task.close()
        continue

print("\nВСЕ ЭКСПЕРИМЕНТЫ ЗАВЕРШЕНЫ!")